# 24 Eval Audio Heads

**Purpose:** Score the saved audio heads on val and check that `AudioBranch` gives the same probabilities as the cached embeddings.

**Inputs:** `models/audio/heads/` (notebook 23), cached BEATs embeddings for val, BEATs weights (for the `AudioBranch` check).

**Outputs:** Tables and plots only; nothing is saved.

**Splits used:** val only (Act_Env, PlayPanT, self-labeled, EmotionalCanines); test: not used.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.metrics import classification_report, f1_score, log_loss

from woofdata.loader.audio.segment import load_audio_segment
from woofdata.tasks import task_rows
from woofdecoder.audio import (
    AROUSAL_HEAD_FILENAME,
    NUDGE_FILENAME,
    SITUATION_HEAD_FILENAME,
    VALENCE_HEAD_FILENAME,
    AudioBranch,
    EmotionNudge,
    LinearHead,
    emotion_inputs,
    encoder_from_config,
    select_layer,
)
from woofdecoder.audio.branch import to_categories_order
from woofdecoder.config import REPO_ROOT, load_config
from woofdecoder.preprocess.segments import BarkSegment
from woofdecoder.taxonomy import CATEGORIES
from wooftrainer.audio_trainer.embeddings import attach_labels, load_embeddings
from wooftrainer.plots import plot_confusion_matrix

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 20)

In [2]:
ENCODER = "beats_iter3_plus_as2m"
LAYER = "mean"  # notebook 20
HEADS_DIR = REPO_ROOT / "models" / "audio" / "heads"

## Load Saved Heads

Load the four saved head files.

In [3]:
class SavedHeads:
    def __init__(self, directory):
        self.situation = LinearHead.load(directory / SITUATION_HEAD_FILENAME)
        self.arousal = LinearHead.load(directory / AROUSAL_HEAD_FILENAME)
        self.valence = LinearHead.load(directory / VALENCE_HEAD_FILENAME)
        self.nudge = EmotionNudge.load(directory / NUDGE_FILENAME)


heads = SavedHeads(HEADS_DIR)
print("situation T:", round(heads.situation.temperature, 3), "(source-balanced run ≈ 1.227)")
print("layer:", heads.situation.layer, "| encoder:", heads.situation.encoder)
print("nudge used:", bool((heads.nudge.weights != 0).any()))

situation T: 1.227 (source-balanced run ≈ 1.227)
layer: mean | encoder: beats_iter3_plus_as2m
nudge used: False


## Helper Functions

Situation probabilities with the nudge, and a bootstrap CI for macro-F1.

In [4]:
def situation_probs(features):
    logits = heads.situation.logits(features)
    shift = heads.nudge.shift(emotion_inputs(features, heads.arousal, heads.valence))
    return heads.situation.probs_from_logits(logits + shift)


def bootstrap_ci(labels, predicted, repeats=1000, seed=0):
    rng = np.random.default_rng(seed)
    scores = []
    for _ in range(repeats):
        index = rng.integers(0, len(labels), len(labels))
        scores.append(f1_score(labels[index], predicted[index], average="macro"))
    return np.percentile(scores, [2.5, 97.5])

## Evaluation on Situation Head

Predict every `audio_context` val window, then look at each source.

In [5]:
x, table = load_embeddings("audio_context", "val", ENCODER)
table = attach_labels(table, "audio_context", "val")
probs = situation_probs(select_layer(x, LAYER))
classes = heads.situation.classes
table["predicted"] = np.array(classes)[probs.argmax(axis=1)]

### PlayPanT: playful_excited only (lab recordings, a third source)

In [6]:
play = (table["source"] == "playpant").to_numpy()
recall = (table.loc[play, "predicted"] == "playful_excited").mean()
mean_p = probs[play, classes.index("playful_excited")].mean()
print(
    f"playpant: {play.sum()} windows | recall {recall:.2f} | mean p(playful_excited) {mean_p:.2f}"
)
print(table.loc[play, "predicted"].value_counts(normalize=True).round(2).to_string())

playpant: 403 windows | recall 0.63 | mean p(playful_excited) 0.38
predicted
playful_excited      0.63
other                0.34
distressed_lonely    0.02
attention_seeking    0.01


**SHORTCUT IS BROKEN!**

### Self-labeled: unseen YouTube dogs, all 5 categories

In [7]:
# Self-labeled: unseen YouTube dogs, all 5 categories
mine = (table["source"] == "self_labeled").to_numpy()
mine_true = table.loc[mine, "mapped_category"].to_numpy()
mine_pred = table.loc[mine, "predicted"].to_numpy()
print(f"\nself-labeled val: {mine.sum()} windows")
print(classification_report(mine_true, mine_pred, labels=classes, digits=3, zero_division=0))
plot_confusion_matrix(mine_true, mine_pred, CATEGORIES, "Situation head, self-labeled val windows")
plt.show()


self-labeled val: 38 windows
                   precision    recall  f1-score   support

 alert_aggressive      1.000     0.500     0.667         2
attention_seeking      0.000     0.000     0.000         4
distressed_lonely      1.000     0.931     0.964        29
            other      0.000     0.000     0.000         2
  playful_excited      0.167     1.000     0.286         1

         accuracy                          0.763        38
        macro avg      0.433     0.486     0.383        38
     weighted avg      0.820     0.763     0.779        38

predicted          alert_aggressive  attention_seeking  distressed_lonely  other  playful_excited
true                                                                                             
alert_aggressive                  1                  0                  0      0                1
attention_seeking                 0                  0                  0      0                4
distressed_lonely                 0         

In v1, the shortcut made YouTube clips of other categories look distressed. Here, on new dogs, none did. Together with the out-of-fold drop from 74% to 31%, that’s good evidence the source balancing worked.

### Self-labeled val per video

In [8]:
video_probs = pd.DataFrame(probs[mine], columns=classes)
video_probs["video"] = table.loc[mine, "clip_id"].str.split("@").str[0].to_numpy()
video_probs["true"] = mine_true
aggregations = {"true": "first"}
for class_name in classes:
    aggregations[class_name] = "mean"
per_video = video_probs.groupby("video").agg(aggregations)
per_video["predicted"] = per_video[classes].idxmax(axis=1)
print(per_video.round(2).to_string())
print(
    "\nvideos correct:", (per_video["predicted"] == per_video["true"]).sum(), "of", len(per_video)
)

                                          true  alert_aggressive  attention_seeking  distressed_lonely  other  playful_excited          predicted
video                                                                                                                                            
self_labeled/audio/A002.mp4  distressed_lonely              0.04               0.08               0.40   0.37             0.11  distressed_lonely
self_labeled/audio/A010.mp4  distressed_lonely              0.10               0.14               0.41   0.15             0.21  distressed_lonely
self_labeled/audio/A011.mp4  distressed_lonely              0.06               0.10               0.45   0.24             0.14  distressed_lonely
self_labeled/audio/A015.mp4  distressed_lonely              0.02               0.11               0.47   0.31             0.08  distressed_lonely
self_labeled/audio/A019.mp4  distressed_lonely              0.06               0.12               0.60   0.15             0.

## Evaluation on Emotion Heads (EmotionalCanines Val)

Arousal and valence macro-F1 with bootstrap CIs.

In [9]:
ex, etable = load_embeddings("audio_emotion", "val", ENCODER)
etable = attach_labels(etable, "audio_emotion", "val")
features = select_layer(ex, LAYER)

for target, head in (("arousal", heads.arousal), ("valence", heads.valence)):
    labels = etable[target].to_numpy()
    p = head.predict_proba(features)
    predicted = np.array(head.classes)[p.argmax(axis=1)]
    low, high = bootstrap_ci(labels, predicted)
    print(
        f"{target}: {len(labels)} clips | macro-F1 {f1_score(labels, predicted, average='macro'):.3f} [{low:.3f}, {high:.3f}] | log-loss {log_loss(labels, p, labels=head.classes):.3f}"
    )

arousal: 200 clips | macro-F1 0.500 [0.431, 0.565] | log-loss 0.996
valence: 200 clips | macro-F1 0.384 [0.314, 0.450] | log-loss 1.070


## Check AudioBranch

Run the real `AudioBranch` on a few val clips and compare with the cached path.

In [10]:
encoder = encoder_from_config(load_config())
branch = AudioBranch(encoder, heads.situation, heads.arousal, heads.valence, heads.nudge, LAYER)

val_rows = task_rows("audio_context", "val")
picked = pd.concat(
    [
        val_rows[val_rows["source"] == "self_labeled"].head(3),
        val_rows[val_rows["source"] == "playpant"].head(2),
    ]
)
for _, row in picked.iterrows():
    audio = load_audio_segment(row, max_seconds=10.0)
    segment = BarkSegment(
        start_s=0.0,
        end_s=len(audio) / 16_000,
        audio=audio,
        sample_rate=16_000,
        source_id=row["clip_id"],
    )
    output = branch.predict(segment)
    cached = probs[table.index[table["clip_id"] == row["clip_id"]][0]]  # notebook 24, head order
    from_cache = to_categories_order(cached, heads.situation.classes)
    print(f"\n{row['clip_id']}  (true: {row['mapped_category']})")
    print("  branch:", {k: round(v, 3) for k, v in output.as_dict().items()})
    print(
        "  top:",
        output.extras["top_category"],
        "| arousal:",
        {k: round(v, 2) for k, v in output.extras["arousal"].items()},
    )
    print(
        f"  largest difference vs notebook 24 (cached embeddings): {np.abs(output.probs - from_cache).max():.4f}"
    )

empty = branch.predict(BarkSegment(start_s=0.0, end_s=1.0, audio=None))
print("\nno audio → present:", empty.present, "| probs:", empty.probs.round(2))

c:\Users\rlaeh\dev\dog-vocal-interpreter\.venv\Lib\site-packages\torch\nn\utils\weight_norm.py:145: FutureWarning: `torch.nn.utils.weight_norm` is deprecated in favor of `torch.nn.utils.parametrizations.weight_norm`.
  WeightNorm.apply(module, name, dim)



self_labeled/audio/A002.mp4@13.000  (true: distressed_lonely)
  branch: {'alert_aggressive': 0.051, 'playful_excited': 0.122, 'distressed_lonely': 0.215, 'attention_seeking': 0.112, 'other': 0.5}
  top: other | arousal: {'High': 0.17, 'Low': 0.56, 'Medium': 0.27}
  largest difference vs notebook 24 (cached embeddings): 0.0000

self_labeled/audio/A002.mp4@3.000  (true: distressed_lonely)
  branch: {'alert_aggressive': 0.02, 'playful_excited': 0.102, 'distressed_lonely': 0.577, 'attention_seeking': 0.054, 'other': 0.248}
  top: distressed_lonely | arousal: {'High': 0.41, 'Low': 0.37, 'Medium': 0.22}
  largest difference vs notebook 24 (cached embeddings): 0.0000

self_labeled/audio/A010.mp4@0.000  (true: distressed_lonely)
  branch: {'alert_aggressive': 0.077, 'playful_excited': 0.244, 'distressed_lonely': 0.377, 'attention_seeking': 0.109, 'other': 0.193}
  top: distressed_lonely | arousal: {'High': 0.5, 'Low': 0.25, 'Medium': 0.25}
  largest difference vs notebook 24 (cached embedding

## Results and decisions

### V1

Shortcut is serious problem. Playpant data are all leaned towards distressed_lonely as it learned by sources not the sound itself.

### V2

distressed_lonely is the strongest audio signal (val precision 1.00). The other four situations are only weakly separated by sound (Act_Env recall ~0.3–0.4; other/playful YouTube clips still lean toward distressed, and toward each other). Fusion should rely on body/scene to separate alert, attention, playful and other. extras will carry arousal (usable) and valence (≈ chance, don’t display).